In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

# =========================
# FILES
# =========================
test_csv = "/kaggle/input/datasets/abaiyan/test-dataset/test_300.csv"
output_csv = "tripadvisor_test_predictions.csv"

# =========================
# LABEL NORMALIZATION
# =========================
def normalize_true_label(label):
    label = str(label).strip().lower()
    if label in ["positive", "postive"]:
        return "Positive"
    elif label == "negative":
        return "Negative"
    elif label == "neutral":
        return "Neutral"
    return None

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(test_csv)

# Ensure required columns
if "Title" not in df.columns:
    df["Title"] = ""
if "Text" not in df.columns:
    raise ValueError("CSV must contain a 'Text' column.")

df["Title"] = df["Title"].fillna("").astype(str)
df["Text"] = df["Text"].fillna("").astype(str)

# Handle labels if available
has_true_labels = "Sentiment" in df.columns
if has_true_labels:
    df["Sentiment"] = df["Sentiment"].apply(normalize_true_label)
    df = df[df["Sentiment"].isin(["Positive", "Neutral", "Negative"])].copy()

# Combine text
df["combined_text"] = (df["Title"].str.strip() + " " + df["Text"].str.strip()).str.strip()

# =========================
# MODEL
# =========================
model_name = "gosorio/robertaSentimentFT_TripAdvisor"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

idx_to_label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

# =========================
# PREDICTION FUNCTION
# =========================
def predict_tripadvisor(text, max_length=512, stride=128):
    if not isinstance(text, str) or text.strip() == "":
        return "Neutral"

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    all_probs = []

    with torch.no_grad():
        for i in range(input_ids.size(0)):
            outputs = model(
                input_ids=input_ids[i].unsqueeze(0),
                attention_mask=attention_mask[i].unsqueeze(0)
            )
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
            all_probs.append(probs)

    avg_probs = np.mean(all_probs, axis=0)
    pred_idx = int(np.argmax(avg_probs))

    return idx_to_label[pred_idx]

# =========================
# RUN PREDICTIONS
# =========================
preds = []

for text in df["combined_text"]:
    pred = predict_tripadvisor(text)
    preds.append(pred)

df["predicted sentiment"] = preds

# Save output
df.drop(columns=["combined_text"]).to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

# =========================
# METRICS
# =========================
if has_true_labels:
    y_true = df["Sentiment"]
    y_pred = df["predicted sentiment"]
    labels = ["Positive", "Neutral", "Negative"]

    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    print("\nAccuracy:", round(acc, 4))
    print("Macro Precision:", round(macro_p, 4))
    print("Macro Recall:", round(macro_r, 4))
    print("Macro F1:", round(macro_f1, 4))
    print("Weighted Precision:", round(weighted_p, 4))
    print("Weighted Recall:", round(weighted_r, 4))
    print("Weighted F1:", round(weighted_f1, 4))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, labels=labels, digits=4, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred, labels=labels))
else:
    print("\nNo 'Sentiment' column found. Metrics skipped.")

In [ ]:
import random
import sys
import pandas as pd
import numpy as np
import torch
import transformers
import sklearn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

# =========================
# REPRODUCIBILITY
# =========================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Optional: stricter determinism
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

# =========================
# FILES
# =========================
test_csv = "/kaggle/input/datasets/abaiyan/test-dataset/test_300.csv"
output_csv = "cardiff_pretrained_test_predictions.csv"

# =========================
# LABEL NORMALIZATION
# =========================
def normalize_true_label(label):
    label = str(label).strip().lower()
    if label in ["positive", "postive"]:
        return "Positive"
    elif label == "negative":
        return "Negative"
    elif label == "neutral":
        return "Neutral"
    return None

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(test_csv)

if "Title" not in df.columns:
    df["Title"] = ""
if "Text" not in df.columns:
    raise ValueError("CSV must contain a 'Text' column.")

df["Title"] = df["Title"].fillna("").astype(str)
df["Text"] = df["Text"].fillna("").astype(str)

has_true_labels = "Sentiment" in df.columns
if has_true_labels:
    df["Sentiment"] = df["Sentiment"].apply(normalize_true_label)
    df = df[df["Sentiment"].isin(["Positive", "Neutral", "Negative"])].copy()

df["combined_text"] = (df["Title"].str.strip() + " " + df["Text"].str.strip()).str.strip()

# =========================
# MODEL
# =========================
model_name = "cardiffnlp/twitter-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Official mapping:
# 0 -> Negative, 1 -> Neutral, 2 -> Positive
idx_to_label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

# =========================
# ENVIRONMENT INFO
# =========================
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("Device:", device)
print("Seed:", SEED)

# =========================
# PREDICTION FUNCTION
# =========================
def predict_cardiff(text, max_length=512, stride=128):
    if not isinstance(text, str) or text.strip() == "":
        return "Neutral"

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    all_probs = []

    with torch.no_grad():
        for i in range(input_ids.size(0)):
            outputs = model(
                input_ids=input_ids[i].unsqueeze(0),
                attention_mask=attention_mask[i].unsqueeze(0)
            )
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
            all_probs.append(probs)

    avg_probs = np.mean(all_probs, axis=0)
    pred_idx = int(np.argmax(avg_probs))

    return idx_to_label[pred_idx]

# =========================
# RUN PREDICTIONS
# =========================
preds = []

for text in df["combined_text"]:
    pred = predict_cardiff(text)
    preds.append(pred)

df["predicted sentiment"] = preds

df.drop(columns=["combined_text"]).to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

# =========================
# METRICS
# =========================
if has_true_labels:
    y_true = df["Sentiment"]
    y_pred = df["predicted sentiment"]
    labels = ["Positive", "Neutral", "Negative"]

    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    print("\nAccuracy:", round(acc, 4))
    print("Macro Precision:", round(macro_p, 4))
    print("Macro Recall:", round(macro_r, 4))
    print("Macro F1:", round(macro_f1, 4))
    print("Weighted Precision:", round(weighted_p, 4))
    print("Weighted Recall:", round(weighted_r, 4))
    print("Weighted F1:", round(weighted_f1, 4))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, labels=labels, digits=4, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred, labels=labels))
else:
    print("\nNo 'Sentiment' column found. Metrics skipped.")

In [ ]:
# =========================
# FINAL CORRECTED CODE
# Fine-tune Cardiff RoBERTa on 700 reviews with chunked training
# Test on separate 300 reviews with chunked inference + probability averaging
# =========================

# Install once if needed:
# !pip install -q transformers datasets accelerate scikit-learn sentencepiece

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

# =========================
# REPRODUCIBILITY
# =========================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Keep this lighter for Kaggle GPU speed
torch.backends.cudnn.benchmark = True

set_seed(SEED)

# =========================
# CONFIG
# =========================
TRAIN_CSV = "/kaggle/input/datasets/abaiyan/train-dataset/train_700.csv"
TEST_CSV = "/kaggle/input/datasets/abaiyan/test-dataset/test_300.csv"

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

MAX_LENGTH_TRAIN = 256
STRIDE_TRAIN = 64

MAX_LENGTH_TEST = 512
STRIDE_TEST = 128

NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

OUTPUT_DIR = "./finetuned_cardiff_chunked"
FINAL_TEST_OUTPUT_CSV = "finetuned_cardiff_chunked_test_300_predictions.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# LABEL MAPS
# =========================
label2id = {"Negative": 0, "Neutral": 1, "Positive": 2}
id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

# =========================
# LABEL NORMALIZATION
# =========================
def normalize_label(x):
    x = str(x).strip().lower()
    if x in ["positive", "postive"]:
        return "Positive"
    elif x == "neutral":
        return "Neutral"
    elif x == "negative":
        return "Negative"
    return None

# =========================
# LOAD DATA
# =========================
def load_and_prepare(csv_path):
    df = pd.read_csv(csv_path)

    if "Title" not in df.columns:
        df["Title"] = ""
    if "Text" not in df.columns:
        raise ValueError(f"{csv_path} must contain 'Text'")
    if "Sentiment" not in df.columns:
        raise ValueError(f"{csv_path} must contain 'Sentiment'")

    df["Title"] = df["Title"].fillna("").astype(str)
    df["Text"] = df["Text"].fillna("").astype(str)
    df["Sentiment"] = df["Sentiment"].apply(normalize_label)

    df = df[df["Sentiment"].isin(label2id.keys())].copy()

    df["combined_text"] = (df["Title"].str.strip() + " " + df["Text"].str.strip()).str.strip()
    df = df[df["combined_text"].str.len() > 0].copy()

    df["label"] = df["Sentiment"].map(label2id)
    return df.reset_index(drop=True)

train_df = load_and_prepare(TRAIN_CSV)
test_df = load_and_prepare(TEST_CSV)

print("\nTrain shape:", train_df.shape)
print(train_df["Sentiment"].value_counts())

print("\nTest shape:", test_df.shape)
print(test_df["Sentiment"].value_counts())

# =========================
# STRATIFIED SPLIT (80-20)
# =========================
train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=SEED
)

train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

print("\nTrain split label counts:")
print(train_part["Sentiment"].value_counts())

print("\nValidation split label counts:")
print(val_part["Sentiment"].value_counts())

# =========================
# TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# =========================
# CHUNK TRAIN/VAL DATA
# =========================
def expand_to_chunks(df, tokenizer, max_length=256, stride=64):
    rows = []

    for _, row in df.iterrows():
        text = row["combined_text"]
        label = int(row["label"])

        encoded = tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            stride=stride,
            return_overflowing_tokens=True,
            return_attention_mask=True
        )

        for i in range(len(encoded["input_ids"])):
            rows.append({
                "input_ids": encoded["input_ids"][i],
                "attention_mask": encoded["attention_mask"][i],
                "labels": label
            })

    return pd.DataFrame(rows)

train_chunks_df = expand_to_chunks(
    train_part,
    tokenizer,
    max_length=MAX_LENGTH_TRAIN,
    stride=STRIDE_TRAIN
)

val_chunks_df = expand_to_chunks(
    val_part,
    tokenizer,
    max_length=MAX_LENGTH_TRAIN,
    stride=STRIDE_TRAIN
)

print("\nExpanded train chunks shape:", train_chunks_df.shape)
print("Expanded val chunks shape:", val_chunks_df.shape)

# =========================
# DATASETS
# =========================
train_ds = Dataset.from_pandas(train_chunks_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_chunks_df, preserve_index=False)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================
# METRICS
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1
    }

# =========================
# CLASS WEIGHTS
# Compute on original train reviews, not chunk counts
# =========================
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_part["label"].values
)
weights = torch.tensor(weights, dtype=torch.float)
print("\nClass weights:", weights)

# =========================
# MODEL
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# =========================
# CUSTOM TRAINER
# =========================
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(weight=weights.to(logits.device))
        loss = loss_fct(logits.view(-1, 3), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# =========================
# TRAINING ARGS
# =========================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
    save_total_limit=1
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# =========================
# TRAIN
# =========================
trainer.train()

# Save model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("\nSaved fine-tuned model to:", OUTPUT_DIR)

# =========================
# CHUNKED INFERENCE ON TEST SET
# =========================
def predict_review_with_chunk_averaging(text, model, tokenizer, max_length=512, stride=128):
    if not isinstance(text, str) or text.strip() == "":
        return "Neutral"

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    all_probs = []

    model.eval()
    with torch.no_grad():
        for i in range(input_ids.size(0)):
            outputs = model(
                input_ids=input_ids[i].unsqueeze(0),
                attention_mask=attention_mask[i].unsqueeze(0)
            )
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
            all_probs.append(probs)

    avg_probs = np.mean(all_probs, axis=0)
    pred_idx = int(np.argmax(avg_probs))
    return id2label[pred_idx]

# =========================
# RUN FINAL TEST PREDICTIONS
# =========================
final_test_preds = []

for text in test_df["combined_text"]:
    pred = predict_review_with_chunk_averaging(
        text=text,
        model=trainer.model,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH_TEST,
        stride=STRIDE_TEST
    )
    final_test_preds.append(pred)

test_df["predicted sentiment"] = final_test_preds

# =========================
# TEST METRICS (REVIEW LEVEL)
# =========================
y_true = test_df["Sentiment"]
y_pred = test_df["predicted sentiment"]
labels = ["Positive", "Neutral", "Negative"]

acc = accuracy_score(y_true, y_pred)

macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n========== FINAL TEST RESULTS ON 300 REVIEWS ==========")
print("Accuracy:", round(acc, 4))
print("Macro Precision:", round(macro_p, 4))
print("Macro Recall:", round(macro_r, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted Precision:", round(weighted_p, 4))
print("Weighted Recall:", round(weighted_r, 4))
print("Weighted F1:", round(weighted_f1, 4))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=labels, digits=4, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=labels))

# =========================
# SAVE
# =========================
test_df.drop(columns=["combined_text"]).to_csv(FINAL_TEST_OUTPUT_CSV, index=False)
print("\nSaved:", FINAL_TEST_OUTPUT_CSV)

In [ ]:
# =========================
# FINAL REPRODUCIBLE ROBERTA-BASE CODE (CORRECTED)
# Fine-tune roberta-base on 700 reviews
# 80/20 stratified train/validation split
# Chunk long reviews during training
# Chunk + average probabilities during testing on 300 reviews
# =========================

# Install once if needed:
# !pip install -q -U transformers datasets accelerate scikit-learn sentencepiece

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import random
import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

# =========================
# REPRODUCIBILITY
# =========================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# For better reproducibility, use deterministic settings
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

set_seed(SEED)

# =========================
# CONFIG
# =========================
TRAIN_CSV = "/kaggle/input/datasets/abaiyan/train-dataset/train_700.csv"
TEST_CSV = "/kaggle/input/datasets/abaiyan/test-dataset/test_300.csv"

MODEL_NAME = "roberta-base"

MAX_LENGTH_TRAIN = 256
STRIDE_TRAIN = 64

MAX_LENGTH_TEST = 512
STRIDE_TEST = 128

NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

OUTPUT_DIR = "./finetuned_roberta_base_chunked"
FINAL_TEST_OUTPUT_CSV = "roberta_base_chunked_test_300_predictions.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

# =========================
# LABEL MAPS
# =========================
label2id = {"Negative": 0, "Neutral": 1, "Positive": 2}
id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

# =========================
# LABEL NORMALIZATION
# =========================
def normalize_label(x):
    x = str(x).strip().lower()
    if x in ["positive", "postive"]:
        return "Positive"
    elif x == "neutral":
        return "Neutral"
    elif x == "negative":
        return "Negative"
    return None

# =========================
# LOAD AND PREPARE
# =========================
def load_and_prepare(csv_path):
    df = pd.read_csv(csv_path)

    if "Title" not in df.columns:
        df["Title"] = ""
    if "Text" not in df.columns:
        raise ValueError(f"{csv_path} must contain a 'Text' column.")
    if "Sentiment" not in df.columns:
        raise ValueError(f"{csv_path} must contain a 'Sentiment' column.")

    df["Title"] = df["Title"].fillna("").astype(str)
    df["Text"] = df["Text"].fillna("").astype(str)
    df["Sentiment"] = df["Sentiment"].apply(normalize_label)

    df = df[df["Sentiment"].isin(["Negative", "Neutral", "Positive"])].copy()

    df["combined_text"] = (
        df["Title"].str.strip() + " " + df["Text"].str.strip()
    ).str.strip()

    df = df[df["combined_text"].str.len() > 0].copy()
    df["label"] = df["Sentiment"].map(label2id)

    return df.reset_index(drop=True)

train_df = load_and_prepare(TRAIN_CSV)
test_df = load_and_prepare(TEST_CSV)

print("\nTrain shape:", train_df.shape)
print(train_df["Sentiment"].value_counts())

print("\nTest shape:", test_df.shape)
print(test_df["Sentiment"].value_counts())

# =========================
# TRAIN / VALIDATION SPLIT (80-20 STRATIFIED)
# =========================
train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=SEED
)

train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

print("\nTrain split label counts:")
print(train_part["Sentiment"].value_counts())

print("\nValidation split label counts:")
print(val_part["Sentiment"].value_counts())

# =========================
# TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# =========================
# CHUNK EXPANSION FOR TRAIN / VAL
# Each chunk becomes one training example
# =========================
def expand_to_chunks(df, tokenizer, max_length, stride):
    rows = []

    for _, row in df.iterrows():
        text = row["combined_text"]
        label = int(row["label"])

        encoded = tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            stride=stride,
            return_overflowing_tokens=True,
            return_attention_mask=True
        )

        for i in range(len(encoded["input_ids"])):
            rows.append({
                "input_ids": encoded["input_ids"][i],
                "attention_mask": encoded["attention_mask"][i],
                "labels": label
            })

    return pd.DataFrame(rows)

train_chunks_df = expand_to_chunks(
    train_part,
    tokenizer,
    max_length=MAX_LENGTH_TRAIN,
    stride=STRIDE_TRAIN
)

val_chunks_df = expand_to_chunks(
    val_part,
    tokenizer,
    max_length=MAX_LENGTH_TRAIN,
    stride=STRIDE_TRAIN
)

print("\nExpanded train chunks shape:", train_chunks_df.shape)
print("Expanded val chunks shape:", val_chunks_df.shape)

# =========================
# DATASETS
# =========================
train_ds = Dataset.from_pandas(
    train_chunks_df[["input_ids", "attention_mask", "labels"]],
    preserve_index=False
)
val_ds = Dataset.from_pandas(
    val_chunks_df[["input_ids", "attention_mask", "labels"]],
    preserve_index=False
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================
# METRICS
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1
    }

# =========================
# CUSTOM TRAINER WITH CLASS WEIGHTS
# FIXED FOR DATAPARALLEL
# =========================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        logits = outputs.logits if hasattr(outputs, "logits") else outputs["logits"]

        class_weights = self.class_weights.to(logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)

        num_labels = logits.size(-1)
        loss = loss_fct(
            logits.view(-1, num_labels),
            labels.view(-1).to(logits.device)
        )

        return (loss, outputs) if return_outputs else loss

# =========================
# CLASS WEIGHTS
# Compute on original train reviews, not chunk rows
# =========================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_part["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

print("\nClass weights:", class_weights)

# =========================
# MODEL
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# =========================
# TRAINING ARGUMENTS
# =========================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# =========================
# TRAIN
# =========================
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nSaved fine-tuned model to:", OUTPUT_DIR)

# =========================
# TEST-TIME CHUNKING + AVERAGING
# =========================
def predict_review_with_chunk_averaging(text, model, tokenizer, max_length, stride, device):
    if not isinstance(text, str) or text.strip() == "":
        return "Neutral"

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    all_probs = []

    model.eval()
    with torch.no_grad():
        for i in range(input_ids.size(0)):
            outputs = model(
                input_ids=input_ids[i].unsqueeze(0),
                attention_mask=attention_mask[i].unsqueeze(0)
            )
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
            all_probs.append(probs)

    avg_probs = np.mean(all_probs, axis=0)
    pred_idx = int(np.argmax(avg_probs))
    return id2label[pred_idx]

# =========================
# RUN FINAL TEST PREDICTIONS
# =========================
final_test_preds = []

# Use the best loaded model from trainer
trained_model = trainer.model.to(device)

for text in test_df["combined_text"]:
    pred = predict_review_with_chunk_averaging(
        text=text,
        model=trained_model,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH_TEST,
        stride=STRIDE_TEST,
        device=device
    )
    final_test_preds.append(pred)

test_df["predicted sentiment"] = final_test_preds

# =========================
# REVIEW-LEVEL TEST METRICS
# =========================
y_true = test_df["Sentiment"]
y_pred = test_df["predicted sentiment"]
labels = ["Positive", "Neutral", "Negative"]

acc = accuracy_score(y_true, y_pred)

macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n========== FINAL TEST RESULTS ON 300 REVIEWS ==========")
print("Accuracy:", round(acc, 4))
print("Macro Precision:", round(macro_p, 4))
print("Macro Recall:", round(macro_r, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted Precision:", round(weighted_p, 4))
print("Weighted Recall:", round(weighted_r, 4))
print("Weighted F1:", round(weighted_f1, 4))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=labels, digits=4, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=labels))

# =========================
# SAVE TEST PREDICTIONS
# =========================
test_df.drop(columns=["combined_text"]).to_csv(FINAL_TEST_OUTPUT_CSV, index=False)
print(f"\nSaved: {FINAL_TEST_OUTPUT_CSV}")